# FIFA World Cup 2026 — Scoreboard Prediction

### Goal

Predict the final scoreline (home goals vs. away goals) for matches at the 2026 FIFA World Cup and using historical international results.

### Data

- **`df_results`** — international match results, 1872–present (`datasets/international_results-master/results.csv`), the historical record of goals scored by each side in each match.
- **`df_formernames`** — historical name changes for national teams (`datasets/international_results-master/former_names.csv`), used to reconcile teams that have been renamed over time so history isn't split across two identities.
- **`df_elo`** — Elo ratings for World Cup 2026 squads (`datasets/elo-ratings/elo_ratings_wc2026.csv`), a single strength number per team going into the tournament.

### Approach

1. Clean and merge the datasets above into one match-level table, with each
   row describing a historical match and the pre-match strength of both sides.
2. Engineer features available *before kickoff* (team Elo/rating, recent
   form, head-to-head history) — never anything that leaks the result.
3. Model the two scores (home goals, away goals) — e.g. independent Poisson
   regressions per side — and evaluate against held-out historical matches
   using a time-based split.
4. Apply the trained model to the 2026 World Cup fixtures to predict a
   scoreboard for each match.

In [1]:
import pandas as pd
import numpy as np
import sklearn as skl

## Datasets analysis
Four raw sources are loaded, kept separate here, and will be merged later into
one model-ready table.

In [2]:
df_elo = pd.read_csv('datasets/elo-ratings/elo_ratings_wc2026.csv')
df_elo = df_elo[df_elo['year'] != 2026]

In [4]:
df_formernames = pd.read_csv('datasets/international_results-master/former_names.csv')

In [5]:
df_results = pd.read_csv('datasets/international_results-master/results.csv')
is_2026_wc = (df_results['tournament'] == 'FIFA World Cup') & (df_results['date'].str.startswith('2026'))
df_results = df_results[~is_2026_wc]

* `df_results`

International match results, 1872–present.
Columns: `date, home_team, away_team, home_score, away_score, tournament, city, country, neutral`. This is the core training table — one row per historical match.

**Filtering:** drop the rows for the **2026 FIFA World Cup** itself
(group stage through the final, `tournament == 'FIFA World Cup'` and
`date` in 2026). These matches can't be paired with a valid pre-match Elo rating (see `df_elo` below) and the tournament is otherwise out of scope for this version. All other 2026 matches (friendlies, qualifiers,
continental cups) and every earlier World Cup (1930–2022) are kept.

* `df_formernames`

Columns: `current, former, start_date, end_date`.
Maps old team names (e.g. Dahomey → Benin, West Germany → Germany) to their current name, so a team's history isn't split across two identities when merging with other sources.

* `df_elo`

Columns include: `year, snapshot_date, country, rating, rank, confederation, is_host, matches_total, wins, losses, draws, goals_for, goals_against, wc2026_exit_round`. One row per country **per year** (1901–2026, 48 countries) — a genuine time series, not a single snapshot, so it can be joined *as of* a match's date to give a leakage-free, pre-match strength feature for both teams.

**Filtering:** drop all **2026** rows. There's no pre-tournament snapshot for that year — only `2026-07-07` (mid-tournament, ratings already reflect games played) and `2026-12-31` (after the tournament is over). Neither is a valid "before kickoff" rating, so 2026 is dropped entirely; the last usable snapshot is `2025-12-31`.

### Making main data set